# Bayesian Sequential Scorer

Two versions of a **rule-based, interpretable model** that scores fly PER traces 0–5:

**Version A: Supervised** — learns rules from human-scored training data
**Version B: Unsupervised** — discovers rules from signal statistics alone (no labels!)

Both follow the same sequential decision logic:
1. **Did the fly react at all?** Compare during-odor signal to before-odor baseline
2. **If no reaction → Score 0**
3. **If reaction, how strong?** Measure magnitude, frequency shift, persistence
4. **Score 1–5** based on thresholds (learned or discovered)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sig
from scipy.fft import fft, fftfreq
from scipy.optimize import minimize
from scipy.special import expit  # sigmoid
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GroupShuffleSplit
import warnings, joblib, os
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
FPS = 40

SCORE_COLORS = {0:'#3274A1',1:'#5EA4C1',2:'#8DC63F',3:'#F5C542',4:'#E8912D',5:'#E23E28'}

## 1. Load & Prepare Data

In [ ]:
DATASET_MAP = {
    'opto_3-oct':'3OCT-Training','opto_ACV':'ACV-Training','opto_AIR':'AIR-Training',
    'Benz_control':'Benz-Control','opto_benz_1':'Benz-Training',
    'opto_benz_1-flagged':'Benz-Training-flagged','EB_control':'EB-Control',
    'opto_EB':'EB-Training','opto_EB(6-training)':'EB-Training(No-Operant)',
    'opto_EB-flagged':'EB-Training-flagged','hex_control':'Hex-Control',
    'hex_control-flagged':'Hex-Control-flagged','opto_hex':'Hex-Training',
    'opto_hex-flagged':'Hex-Training-flagged',
}
TRACE_PATTERN = r'^testing_\d+_fly\d+_distances_fly\d+_angle_distance_rms_envelope$'

# Load all traces (scored + unlabeled)
traces_df = pd.read_csv('../data/all_envelope_rows_wide_combined_base.csv')
traces_df['trial_base'] = traces_df['trial_label'].str.extract(r'^(testing_\d+)')
traces_df['trace_fly_num'] = traces_df['trial_label'].str.extract(r'fly(\d+)_')[0].astype(int)
dir_cols = [f'dir_val_{i}' for i in range(3601) if f'dir_val_{i}' in traces_df.columns]

dist_mask = traces_df['trial_label'].str.match(TRACE_PATTERN)
traces_dist = traces_df[dist_mask].copy()
print(f'All distance-type traces: {len(traces_dist)}')

# Load scores
scores_df = pd.read_csv('../data/new_scoring_ordered_complete1.csv')
scores_df['user_score_odor'] = scores_df['user_score_odor'].replace(6, 5)
scores_df['dataset_mapped'] = scores_df['dataset'].map(DATASET_MAP)
scores_df['trial_base'] = scores_df['trial_label']

# Merge scored subset
scored = traces_dist.merge(
    scores_df[['dataset_mapped','fly','fly_number','trial_base','user_score_odor']],
    left_on=['dataset','fly','trace_fly_num','trial_base'],
    right_on=['dataset_mapped','fly','fly_number','trial_base'], how='inner')

print(f'Scored traces: {len(scored)}')
print(scored['user_score_odor'].value_counts().sort_index())

## 2. Measurement Functions

These compute interpretable quantities from each trace — the **evidence** the model reasons over.

In [ ]:
def interpolate_nans(trace):
    out = trace.copy()
    nans = np.isnan(out)
    if not nans.any():
        return out
    x = np.arange(len(out))
    out[nans] = np.interp(x[nans], x[~nans], out[~nans])
    return out

def compute_measurements(trace, fps=40):
    """Compute all interpretable measurements from a single trace."""
    trace = interpolate_nans(trace)
    before = trace[:1200]; during = trace[1200:2400]; after = trace[2400:]
    m = {}
    
    # Basic stats
    m['before_mean'] = np.mean(before)
    m['before_std'] = np.std(before)
    m['during_mean'] = np.mean(during)
    m['during_std'] = np.std(during)
    m['after_mean'] = np.mean(after)
    m['after_std'] = np.std(after)
    
    # Mean shift
    m['mean_shift'] = m['during_mean'] - m['before_mean']
    m['mean_shift_z'] = m['mean_shift'] / (m['before_std'] + 1e-10)
    m['mean_ratio'] = m['during_mean'] / (m['before_mean'] + 1e-10)
    
    # Variability change
    m['std_ratio'] = m['during_std'] / (m['before_std'] + 1e-10)
    
    # Peak response
    m['during_max'] = np.max(during)
    m['peak_over_baseline'] = m['during_max'] / (m['before_mean'] + 1e-10)
    m['peak_z'] = (m['during_max'] - m['before_mean']) / (m['before_std'] + 1e-10)
    
    # AUC
    m['auc_before'] = np.trapz(before) / len(before)
    m['auc_during'] = np.trapz(during) / len(during)
    m['auc_after'] = np.trapz(after) / len(after)
    m['auc_ratio'] = m['auc_during'] / (m['auc_before'] + 1e-10)
    
    # Frequency power shift
    bands = {'vlow':(0,0.5),'low':(0.5,1.0),'mid':(1.0,3.0),'high':(3.0,5.0),'vhigh':(5.0,10.0)}
    for epoch_name, epoch_data in [('before', before), ('during', during)]:
        f_w, pxx = sig.welch(epoch_data - np.mean(epoch_data), fs=fps, nperseg=256, noverlap=128)
        for bn, (fmin, fmax) in bands.items():
            idx = (f_w >= fmin) & (f_w < fmax)
            m[f'{epoch_name}_{bn}_power'] = pxx[idx].sum()
    for bn in bands:
        m[f'power_ratio_{bn}'] = m[f'during_{bn}_power'] / (m[f'before_{bn}_power'] + 1e-10)
    m['total_power_ratio'] = sum(m[f'during_{bn}_power'] for bn in bands) / \
                              (sum(m[f'before_{bn}_power'] for bn in bands) + 1e-10)
    
    # Persistence
    m['after_vs_before_mean'] = m['after_mean'] / (m['before_mean'] + 1e-10)
    m['persistence'] = (m['after_mean'] - m['before_mean']) / (m['during_mean'] - m['before_mean'] + 1e-10)
    
    # Time to peak
    m['time_to_peak_frac'] = np.argmax(during) / len(during)
    
    # Sustained response
    m['frac_above_baseline'] = (during > (m['before_mean'] + 2 * m['before_std'])).mean()
    
    return m

In [ ]:
# Compute measurements for ALL traces (scored + unlabeled)
print('Computing measurements for scored traces...')
scored_meas = []
for _, row in scored.iterrows():
    trace = row[dir_cols].values.astype(float)
    scored_meas.append(compute_measurements(trace))
scored_meas_df = pd.DataFrame(scored_meas).replace([np.inf, -np.inf], np.nan).fillna(0)
scored_meas_df['score'] = scored['user_score_odor'].values
print(f'Scored: {len(scored_meas_df)}')

print('Computing measurements for ALL traces (for unsupervised model)...')
all_meas = []
for _, row in traces_dist.iterrows():
    trace = row[dir_cols].values.astype(float)
    all_meas.append(compute_measurements(trace))
all_meas_df = pd.DataFrame(all_meas).replace([np.inf, -np.inf], np.nan).fillna(0)
print(f'All traces: {len(all_meas_df)}')

## 3. Model Class (GBM-based Sequential Scorer)

In [ ]:
class BayesianSequentialScorer:
    """
    Sequential, interpretable scorer.
    Stage 1: Reaction Gate — did the fly react? (GBM binary classifier)
    Stage 2: Intensity Scorer — how strong 1-5? (GBM ordinal/multiclass)
    
    Uses ALL available measurements but explains via feature contributions.
    Soft gating: P(reacted) weights the final probabilities instead of
    hard-cutting at 0.5 (so Stage 1 errors don't fully cascade).
    """
    
    ALL_FEATURES = [
        'mean_shift_z', 'mean_shift', 'mean_ratio',
        'std_ratio', 'before_std', 'during_std',
        'peak_z', 'peak_over_baseline', 'during_max',
        'auc_ratio', 'auc_before', 'auc_during',
        'total_power_ratio',
        'power_ratio_vlow', 'power_ratio_low', 'power_ratio_mid',
        'power_ratio_high', 'power_ratio_vhigh',
        'before_vlow_power', 'before_low_power', 'before_mid_power',
        'before_high_power', 'before_vhigh_power',
        'during_vlow_power', 'during_low_power', 'during_mid_power',
        'during_high_power', 'during_vhigh_power',
        'frac_above_baseline', 'persistence',
        'after_vs_before_mean', 'time_to_peak_frac',
        'before_mean', 'during_mean', 'after_mean', 'after_std',
    ]
    
    def __init__(self):
        self.gate_model = None       # Stage 1: binary GBM
        self.intensity_model = None  # Stage 2: multiclass GBM (scores 1-5)
        self.mode = None
    
    # ──────── SUPERVISED ────────
    def fit_supervised(self, meas_df, labels, verbose=True):
        from sklearn.ensemble import GradientBoostingClassifier
        self.mode = 'supervised'
        labels = np.array(labels)
        X = meas_df[self.ALL_FEATURES].values.astype(float)
        
        # Stage 1: Reaction gate (binary)
        if verbose: print('SUPERVISED — Stage 1: Training reaction gate (GBM)...')
        binary = (labels > 0).astype(int)
        pos_weight = len(binary) / (2 * binary.sum() + 1e-10)
        neg_weight = len(binary) / (2 * (len(binary) - binary.sum()) + 1e-10)
        sample_w = np.where(binary == 1, pos_weight, neg_weight)
        
        self.gate_model = GradientBoostingClassifier(
            n_estimators=200, max_depth=4, min_samples_leaf=10,
            subsample=0.8, random_state=42)
        self.gate_model.fit(X, binary, sample_weight=sample_w)
        gate_acc = self.gate_model.score(X, binary)
        if verbose:
            print(f'  Gate training accuracy: {gate_acc:.3f}')
            imp = pd.Series(self.gate_model.feature_importances_, index=self.ALL_FEATURES)
            top = imp.nlargest(10)
            print(f'  Top 10 gate features:')
            for f, v in top.items():
                print(f'    {f:.<35} {v:.4f}')
        
        # Stage 2: Intensity (multiclass 1-5, only on positives)
        if verbose: print('\nSUPERVISED — Stage 2: Training intensity scorer (GBM)...')
        pos_mask = labels > 0
        X_pos = X[pos_mask]
        y_pos = labels[pos_mask]
        
        self.intensity_model = GradientBoostingClassifier(
            n_estimators=200, max_depth=4, min_samples_leaf=5,
            subsample=0.8, random_state=42)
        self.intensity_model.fit(X_pos, y_pos)
        int_acc = self.intensity_model.score(X_pos, y_pos)
        if verbose:
            print(f'  Intensity training accuracy: {int_acc:.3f}')
            imp = pd.Series(self.intensity_model.feature_importances_, index=self.ALL_FEATURES)
            top = imp.nlargest(10)
            print(f'  Top 10 intensity features:')
            for f, v in top.items():
                print(f'    {f:.<35} {v:.4f}')
        return self
    
    # ──────── UNSUPERVISED ────────
    def fit_unsupervised(self, meas_df, verbose=True):
        """Discover scoring from signal alone — no labels.
        
        Domain priors (from PER assay biology):
        - ~40-45% of flies react to odor (not 25%)
        - Among reactors, scores are skewed: 1-2 are most common (~50%),
          high scores (4-5) are rarer (~25%)
        """
        from sklearn.ensemble import GradientBoostingClassifier
        self.mode = 'unsupervised'
        X = meas_df[self.ALL_FEATURES].values.astype(float)
        
        if verbose: print('UNSUPERVISED — Stage 1: Building reaction gate from signal stats...')
        
        # Composite signal: weighted combination of key change metrics
        composite = (
            meas_df['mean_shift_z'].values * 1.5 +
            meas_df['peak_z'].values * 1.0 +
            meas_df['frac_above_baseline'].values * 2.0 +
            meas_df['total_power_ratio'].values * 0.8 +
            meas_df['std_ratio'].values * 0.5 +
            meas_df['auc_ratio'].values * 0.5
        )
        
        # Domain prior: ~42% reaction rate (top ~42% of composite scores)
        target_reaction_rate = 0.42
        threshold = np.percentile(composite, (1 - target_reaction_rate) * 100)
        pseudo_binary = (composite > threshold).astype(int)
        
        if verbose: print(f'  Pseudo-reaction rate: {pseudo_binary.mean():.3f} (target: {target_reaction_rate:.2f})')
        
        self.gate_model = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, min_samples_leaf=15,
            subsample=0.8, random_state=42)
        self.gate_model.fit(X, pseudo_binary)
        
        if verbose:
            imp = pd.Series(self.gate_model.feature_importances_, index=self.ALL_FEATURES)
            top = imp.nlargest(8)
            print(f'  Top gate features: {dict(top.round(4))}')
        
        # Stage 2: intensity scores 1-5 with skewed distribution
        if verbose: print('\nUNSUPERVISED — Stage 2: Assigning intensity by percentile rank...')
        gate_probs = self.gate_model.predict_proba(X)[:, 1]
        passed_mask = gate_probs > 0.5
        X_passed = X[passed_mask]
        composite_passed = composite[passed_mask]
        
        # Domain prior: score distribution is right-skewed
        # Score 1: ~27%, Score 2: ~26%, Score 3: ~20%, Score 4: ~14%, Score 5: ~13%
        # Cumulative: 27, 53, 73, 87 → percentile boundaries
        pct = np.percentile(composite_passed, [27, 53, 73, 87])
        pseudo_scores = np.digitize(composite_passed, pct) + 1
        pseudo_scores = np.clip(pseudo_scores, 1, 5)
        
        if verbose:
            print(f'  Passed gate: {passed_mask.sum()}/{len(X)}')
            dist = pd.Series(pseudo_scores).value_counts().sort_index()
            print(f'  Pseudo-score distribution:')
            for s, c in dist.items():
                print(f'    Score {s}: {c:4d} ({c/len(pseudo_scores)*100:.1f}%)')
        
        self.intensity_model = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, min_samples_leaf=10,
            subsample=0.8, random_state=42)
        self.intensity_model.fit(X_passed, pseudo_scores)
        
        return self
    
    # ──────── PREDICTION ────────
    def predict_one(self, measurements):
        """Predict score with full reasoning trace."""
        meas_row = pd.DataFrame([measurements]).replace([np.inf, -np.inf], np.nan).fillna(0)
        X = meas_row[self.ALL_FEATURES].values.astype(float)
        steps = []
        
        # Stage 1: Gate
        p_reacted = float(self.gate_model.predict_proba(X)[0, 1])
        
        gate_evidence = {}
        imp = self.gate_model.feature_importances_
        for i, feat in enumerate(self.ALL_FEATURES):
            raw = float(X[0, i])
            gate_evidence[feat] = {
                'raw_value': raw,
                'importance': float(imp[i]),
            }
        steps.append({'stage': 'Reaction Gate', 'p_reacted': p_reacted,
                      'verdict': 'REACTED' if p_reacted > 0.5 else 'NO REACTION',
                      'evidence': gate_evidence})
        
        # SOFT GATING: P(score=0) = 1-P(reacted), P(score=k) = P(reacted)*P(k|reacted)
        probs = {0: 1 - p_reacted}
        
        # Stage 2: Intensity (always compute, weight by p_reacted)
        int_probs_raw = self.intensity_model.predict_proba(X)[0]
        int_classes = self.intensity_model.classes_
        
        int_evidence = {}
        int_imp = self.intensity_model.feature_importances_
        for i, feat in enumerate(self.ALL_FEATURES):
            int_evidence[feat] = {
                'raw_value': float(X[0, i]),
                'importance': float(int_imp[i]),
            }
        
        conditional_probs = {}
        for cls, p in zip(int_classes, int_probs_raw):
            probs[int(cls)] = p_reacted * float(p)
            conditional_probs[int(cls)] = float(p)
        for k in range(1, 6):
            if k not in probs:
                probs[k] = 0.0
        
        steps.append({'stage': 'Intensity Scorer',
                      'conditional_probs': conditional_probs,
                      'evidence': int_evidence})
        
        # Normalize
        total = sum(probs.values())
        probs = {k: v/total for k, v in probs.items()}
        predicted = max(probs, key=probs.get)
        return predicted, probs, steps
    
    def predict(self, meas_df):
        preds, all_probs, all_steps = [], [], []
        for _, row in meas_df.iterrows():
            pred, probs, steps = self.predict_one(row.to_dict())
            preds.append(pred); all_probs.append(probs); all_steps.append(steps)
        return np.array(preds), all_probs, all_steps


## 4. Train Both Models

In [ ]:
# Train/test split (group by fly)
groups = scored['fly'].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(scored_meas_df, scored_meas_df['score'], groups))

train_meas = scored_meas_df.iloc[train_idx].reset_index(drop=True)
test_meas = scored_meas_df.iloc[test_idx].reset_index(drop=True)
train_labels = scored_meas_df['score'].values[train_idx]
test_labels = scored_meas_df['score'].values[test_idx]

print(f'Train: {len(train_meas)}, Test: {len(test_meas)}')
print(f'Train: {pd.Series(train_labels).value_counts().sort_index().to_dict()}')
print(f'Test:  {pd.Series(test_labels).value_counts().sort_index().to_dict()}')

In [ ]:
# Version A: SUPERVISED
print('='*60)
scorer_sup = BayesianSequentialScorer()
scorer_sup.fit_supervised(train_meas, train_labels, verbose=True)

In [ ]:
# Version B: UNSUPERVISED (uses ALL traces, no labels)
print('='*60)
scorer_unsup = BayesianSequentialScorer()
scorer_unsup.fit_unsupervised(all_meas_df, verbose=True)

## 5. Evaluate Both on Test Set

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Evaluate both models on test set
sup_preds, sup_probs, sup_steps = scorer_sup.predict(test_meas)
unsup_preds, unsup_probs, unsup_steps = scorer_unsup.predict(test_meas)

print('SUPERVISED:')
print(f'  Accuracy (exact): {accuracy_score(test_labels, sup_preds):.3f}')
print(f'  Within ±1: {np.mean(np.abs(sup_preds - test_labels) <= 1):.3f}')
print(f'  Binary accuracy (0 vs >0): {np.mean((sup_preds > 0) == (test_labels > 0)):.3f}')
print()
print('UNSUPERVISED:')
print(f'  Accuracy (exact): {accuracy_score(test_labels, unsup_preds):.3f}')
print(f'  Within ±1: {np.mean(np.abs(unsup_preds - test_labels) <= 1):.3f}')
print(f'  Binary accuracy (0 vs >0): {np.mean((unsup_preds > 0) == (test_labels > 0)):.3f}')
print()
print('SUPERVISED — Classification Report:')
print(classification_report(test_labels, sup_preds, zero_division=0))


In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, preds, title in [(axes[0], sup_preds, 'SUPERVISED'), (axes[1], unsup_preds, 'UNSUPERVISED')]:
    cm = confusion_matrix(test_labels, preds, labels=range(6))
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-10)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(6)); ax.set_yticks(range(6))
    ax.set_xticklabels(range(6)); ax.set_yticklabels(range(6))
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(test_labels, preds):.3f}')
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if cm_norm[i,j] > 0.5 else 'black')

plt.colorbar(im, ax=axes)
fig.suptitle('Confusion Matrices: Supervised vs Unsupervised', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Explain Predictions — Visual Comparison

In [ ]:
def explain_prediction(trace, scorer, trial_name='', true_score=None, fps=40):
    trace = interpolate_nans(trace)
    meas = compute_measurements(trace, fps)
    pred, probs, steps = scorer.predict_one(meas)
    N = len(trace); t_sec = np.arange(N) / fps
    
    fig = plt.figure(figsize=(20, 14))
    title = f'{trial_name}\nPredicted: {pred} ({scorer.mode})'
    if true_score is not None:
        match = 'CORRECT' if pred == true_score else f'OFF BY {abs(pred-true_score)}'
        title += f'  |  True: {true_score}  |  {match}'
    fig.suptitle(title, fontsize=15, fontweight='bold', color=SCORE_COLORS.get(pred, 'black'))
    
    # (A) Raw trace
    ax = fig.add_subplot(3, 3, 1)
    ax.plot(t_sec, trace, linewidth=0.6, color='black')
    ax.axvspan(0, 30, alpha=0.1, color='gray'); ax.axvspan(30, 60, alpha=0.15, color='red'); ax.axvspan(60, 90, alpha=0.1, color='blue')
    ax.axhline(meas['before_mean'], color='gray', linestyle='--', alpha=0.5, label=f'Before: {meas["before_mean"]:.1f}')
    ax.axhline(meas['during_mean'], color='red', linestyle='--', alpha=0.5, label=f'During: {meas["during_mean"]:.1f}')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('dir_val'); ax.set_title('(A) Raw Trace'); ax.legend(fontsize=7)
    
    # (B) Probabilities
    ax = fig.add_subplot(3, 3, 2)
    bars = ax.bar([str(k) for k in range(6)], [probs.get(k, 0) for k in range(6)],
                  color=[SCORE_COLORS[k] for k in range(6)], edgecolor='black')
    if true_score is not None:
        bars[true_score].set_edgecolor('green'); bars[true_score].set_linewidth(3)
    ax.set_xlabel('Score'); ax.set_ylabel('P'); ax.set_title(f'(B) Probabilities -> {pred}'); ax.set_ylim(0, 1)
    
    # (C) Top gate feature importances
    ax = fig.add_subplot(3, 3, 3)
    gate = steps[0]
    gate_imp = pd.Series({f: gate['evidence'][f]['importance'] for f in gate['evidence']})
    top_gate = gate_imp.nlargest(8)
    colors_g = ['#E23E28' if gate['evidence'][f]['raw_value'] > 1 else '#3274A1' for f in top_gate.index]
    top_gate.plot.barh(ax=ax, color=colors_g, edgecolor='black')
    ax.set_title(f'(C) Gate: {gate["verdict"]} (P={gate["p_reacted"]:.3f})\nTop features by importance')
    ax.grid(True, alpha=0.2, axis='x')
    
    # (D) Key measurement values
    ax = fig.add_subplot(3, 3, 4)
    keys = ['mean_shift_z', 'std_ratio', 'peak_z', 'auc_ratio', 'total_power_ratio', 'frac_above_baseline']
    vals = [meas[k] for k in keys]
    colors_meas = ['#E23E28' if v > 1 else '#3274A1' for v in vals]
    ax.barh(keys, vals, color=colors_meas, edgecolor='black')
    ax.axvline(1, color='gray', linestyle='--', alpha=0.5); ax.set_title('(D) Key Measurements')
    ax.grid(True, alpha=0.2, axis='x')
    
    # (E) Top intensity feature importances
    ax = fig.add_subplot(3, 3, 5)
    if len(steps) > 1:
        int_step = steps[1]
        int_imp = pd.Series({f: int_step['evidence'][f]['importance'] for f in int_step['evidence']})
        top_int = int_imp.nlargest(8)
        colors_i = ['#E23E28' if int_step['evidence'][f]['raw_value'] > 1 else '#3274A1' for f in top_int.index]
        top_int.plot.barh(ax=ax, color=colors_i, edgecolor='black')
        cond = int_step.get('conditional_probs', {})
        cond_str = ', '.join(f'{k}:{v:.2f}' for k, v in sorted(cond.items()))
        ax.set_title(f'(E) Intensity features\nP(score|reacted): {cond_str}')
    else:
        ax.text(0.5, 0.5, 'Skipped', ha='center', va='center', fontsize=12)
        ax.set_title('(E) Intensity: Skipped')
    ax.grid(True, alpha=0.2, axis='x')
    
    # (F) PSD comparison
    ax = fig.add_subplot(3, 3, 6)
    f_w, pxx_b = sig.welch(trace[:1200] - np.mean(trace[:1200]), fs=fps, nperseg=256, noverlap=128)
    _, pxx_d = sig.welch(trace[1200:2400] - np.mean(trace[1200:2400]), fs=fps, nperseg=256, noverlap=128)
    ax.semilogy(f_w, pxx_b, label='Before', color='gray', linewidth=1.5)
    ax.semilogy(f_w, pxx_d, label='During', color='red', linewidth=1.5)
    ax.set_xlabel('Hz'); ax.set_ylabel('PSD'); ax.set_title('(F) Before vs During PSD'); ax.set_xlim(0, 10); ax.legend(fontsize=8)
    
    # (G) Reasoning text
    ax = fig.add_subplot(3, 1, 3); ax.axis('off')
    lines = [f'REASONING TRACE ({scorer.mode.upper()})', '\u2500' * 60, '']
    lines.append(f'STEP 1: Before mean={meas["before_mean"]:.2f}+/-{meas["before_std"]:.2f}, During mean={meas["during_mean"]:.2f}+/-{meas["during_std"]:.2f}')
    lines.append(f'  Mean shift: {meas["mean_shift"]:.2f} ({meas["mean_shift_z"]:.2f}sigma), Power ratio: {meas["total_power_ratio"]:.2f}x, Peak z: {meas["peak_z"]:.2f}')
    lines.append(f'  Frac above 2sigma: {meas["frac_above_baseline"]:.3f}, AUC ratio: {meas["auc_ratio"]:.3f}')
    lines.append(f'')
    lines.append(f'STEP 2: Reaction Gate -> P(reacted) = {gate["p_reacted"]:.3f} -> {gate["verdict"]}')
    gate_imp_sorted = sorted(gate['evidence'].items(), key=lambda x: x[1]['importance'], reverse=True)[:5]
    for f, info in gate_imp_sorted:
        lines.append(f'  {f} = {info["raw_value"]:.3f} (importance: {info["importance"]:.4f})')
    if len(steps) > 1:
        cond = steps[1].get('conditional_probs', {})
        lines.append(f'')
        lines.append(f'STEP 3: Intensity -> P(score|reacted): {cond}')
        int_imp_sorted = sorted(steps[1]['evidence'].items(), key=lambda x: x[1]['importance'], reverse=True)[:5]
        for f, info in int_imp_sorted:
            lines.append(f'  {f} = {info["raw_value"]:.3f} (importance: {info["importance"]:.4f})')
    lines.append(f'')
    lines.append(f'FINAL: Score {pred} (P={probs[pred]:.3f})')
    ax.text(0.02, 0.98, '\n'.join(lines), transform=ax.transAxes, fontsize=10,
            fontfamily='monospace', va='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    plt.tight_layout()
    return fig, pred, probs, steps


In [ ]:
# Demo: one trace per true score, both models
for s in range(6):
    idx = np.where(test_labels == s)[0]
    if len(idx) == 0: continue
    di = idx[0]
    row = scored.iloc[test_idx[di]]
    trace = row[dir_cols].values.astype(float)
    name = f'{row["dataset"]} / {row["fly"]} / {row["trial_base"]}'
    
    fig1, p1, _, _ = explain_prediction(trace, scorer_sup, trial_name=f'SUPERVISED: {name}', true_score=s)
    plt.show(); plt.close()
    
    fig2, p2, _, _ = explain_prediction(trace, scorer_unsup, trial_name=f'UNSUPERVISED: {name}', true_score=s)
    plt.show(); plt.close()

## 7. Run Both on Unlabeled Fly

In [ ]:
target = traces_dist[
    (traces_dist['dataset'] == 'Hex-Control-24') &
    (traces_dist['fly'] == 'march_18_batch_2_rig_2') &
    (traces_dist['trace_fly_num'] == 2)
].copy()
target['trial_num'] = target['trial_base'].str.extract(r'testing_(\d+)').astype(int)
target = target.sort_values('trial_num').reset_index(drop=True)

outdir = 'signal_classification_output/bayesian_hex_ctrl24_fly2'
os.makedirs(outdir, exist_ok=True)

results = []
for _, row in target.iterrows():
    tn = int(row['trial_num'])
    trace = row[dir_cols].values.astype(float)
    name = f'Hex-Control-24 / march_18_batch_2_rig_2 / Fly 2 — T{tn}'
    
    fig1, p1, pr1, s1 = explain_prediction(trace, scorer_sup, trial_name=f'SUPERVISED: {name}')
    fig1.savefig(f'{outdir}/T{tn:02d}_supervised.png', dpi=150, bbox_inches='tight'); plt.close()
    
    fig2, p2, pr2, s2 = explain_prediction(trace, scorer_unsup, trial_name=f'UNSUPERVISED: {name}')
    fig2.savefig(f'{outdir}/T{tn:02d}_unsupervised.png', dpi=150, bbox_inches='tight'); plt.close()
    
    results.append({'trial': tn, 'supervised': p1, 'unsupervised': p2,
                    'p_reacted_sup': s1[0]['p_reacted'], 'p_reacted_unsup': s2[0]['p_reacted']})
    print(f'T{tn:>2}: Supervised={p1}, Unsupervised={p2}  '
          f'P(react): sup={s1[0]["p_reacted"]:.3f} unsup={s2[0]["p_reacted"]:.3f}')

print(f'\nFigures: {outdir}/')

In [ ]:
# Summary comparison
res_df = pd.DataFrame(results)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

x = np.arange(len(res_df))
w = 0.35

ax = axes[0]
bars1 = ax.bar(x - w/2, res_df['supervised'], w, label='Supervised',
               color=[SCORE_COLORS[s] for s in res_df['supervised']], edgecolor='black')
bars2 = ax.bar(x + w/2, res_df['unsupervised'], w, label='Unsupervised',
               color=[SCORE_COLORS[s] for s in res_df['unsupervised']], edgecolor='black', hatch='//')
ax.set_xticks(x); ax.set_xticklabels([f'T{t}' for t in res_df['trial']])
ax.set_ylabel('Predicted Score'); ax.set_title('Supervised vs Unsupervised Predictions')
ax.legend(); ax.set_ylim(0, 5.8); ax.grid(True, alpha=0.2, axis='y')
for b, s in zip(bars1, res_df['supervised']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1, str(s), ha='center', fontsize=9, fontweight='bold')
for b, s in zip(bars2, res_df['unsupervised']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1, str(s), ha='center', fontsize=9, fontweight='bold')

ax = axes[1]
ax.bar(x - w/2, res_df['p_reacted_sup'], w, label='Supervised', color='steelblue', edgecolor='black')
ax.bar(x + w/2, res_df['p_reacted_unsup'], w, label='Unsupervised', color='coral', edgecolor='black', hatch='//')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Reaction threshold')
ax.set_xticks(x); ax.set_xticklabels([f'T{t}' for t in res_df['trial']])
ax.set_ylabel('P(reacted)'); ax.set_title('Reaction Probability: Supervised vs Unsupervised')
ax.legend(); ax.set_ylim(0, 1); ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

## 8. Visualize Learned Rules

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for row_idx, (scorer, label) in enumerate([(scorer_sup, 'SUPERVISED'), (scorer_unsup, 'UNSUPERVISED')]):
    # Gate feature importances
    ax = axes[row_idx, 0]
    gate_imp = pd.Series(scorer.gate_model.feature_importances_, index=scorer.ALL_FEATURES)
    top_gate = gate_imp.nlargest(12).sort_values()
    top_gate.plot.barh(ax=ax, color='#3274A1', edgecolor='black')
    ax.set_title(f'{label}\nGate Feature Importances (Top 12)')
    ax.grid(True, alpha=0.2, axis='x')
    ax.set_xlabel('Importance')
    
    # Intensity feature importances
    ax = axes[row_idx, 1]
    int_imp = pd.Series(scorer.intensity_model.feature_importances_, index=scorer.ALL_FEATURES)
    top_int = int_imp.nlargest(12).sort_values()
    top_int.plot.barh(ax=ax, color='#E23E28', edgecolor='black')
    ax.set_title(f'{label}\nIntensity Feature Importances (Top 12)')
    ax.grid(True, alpha=0.2, axis='x')
    ax.set_xlabel('Importance')

fig.suptitle('GBM Feature Importances: Gate vs Intensity', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Print learned rules summary
for scorer, label in [(scorer_sup, 'SUPERVISED'), (scorer_unsup, 'UNSUPERVISED')]:
    print(f'\n{"="*60}')
    print(f'{label} SCORING RULES (GBM-based)')
    print(f'{"="*60}')
    
    print(f'\nSTAGE 1: REACTION GATE — "Did the fly react to odor?"')
    print(f'  GBM with {scorer.gate_model.n_estimators} trees, max_depth={scorer.gate_model.max_depth}')
    gate_imp = pd.Series(scorer.gate_model.feature_importances_, index=scorer.ALL_FEATURES)
    top_gate = gate_imp.nlargest(10)
    print(f'  Top features driving the gate decision:')
    for f, v in top_gate.items():
        print(f'    {f:.<40} importance={v:.4f}')
    
    print(f'\nSTAGE 2: INTENSITY — "How strong was the reaction?" (scores 1-5)')
    print(f'  GBM with {scorer.intensity_model.n_estimators} trees, max_depth={scorer.intensity_model.max_depth}')
    int_imp = pd.Series(scorer.intensity_model.feature_importances_, index=scorer.ALL_FEATURES)
    top_int = int_imp.nlargest(10)
    print(f'  Top features driving intensity scoring:')
    for f, v in top_int.items():
        print(f'    {f:.<40} importance={v:.4f}')
    
    print(f'\n  Intensity classes: {list(scorer.intensity_model.classes_)}')
    print(f'  NOTE: Soft gating — P(reacted) weights the intensity probs,')
    print(f'        so borderline reactions get lower confidence, not hard-cut.')


## 9. Save Models

In [ ]:
save_dir = '../outputs/artifacts/bayesian_sequential_scorer'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(scorer_sup, f'{save_dir}/scorer_supervised.joblib')
joblib.dump(scorer_unsup, f'{save_dir}/scorer_unsupervised.joblib')

print(f'Saved to {save_dir}/')
print(f'\nUsage:')
print(f'  scorer = joblib.load("{save_dir}/scorer_supervised.joblib")')
print(f'  measurements = compute_measurements(trace)')
print(f'  score, probs, reasoning = scorer.predict_one(measurements)')